# PINN-SFR-Transient — safety map & dynamics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppigazzini/pinn-sfr-transient/blob/main/notebooks/02_safety_map.ipynb)

A **parameter-space** companion to
[`01_ulof_walkthrough.ipynb`](01_ulof_walkthrough.ipynb). The first notebook runs
*one* Unprotected Loss of Flow (ULOF) transient; this one asks the engineering
question behind it: **how big does the power excursion get as the design
parameters change, and where does the transient stop being benign?**

Everything here is **numpy/scipy only** — no PyTorch — and runs in well under a
minute on a CPU (a 16×16 sweep is ~250 stiff solves at ~50 ms each). Background:
[`docs/physics_theory.md`](../docs/physics_theory.md).

> Tip: *Run All*. The first cell installs the package automatically on Google
> Colab; each user gets their own Colab runtime (the badge just loads this
> notebook from GitHub).

In [ ]:
# --- setup: make `pinn_sfr_transient` importable whether the package is
#     installed (`uv sync`), run from a repo checkout, or on Google Colab ---
import importlib, pathlib, subprocess, sys

_REPO = "git+https://github.com/ppigazzini/pinn-sfr-transient.git"


def _ensure_package():
    try:
        import pinn_sfr_transient  # noqa: F401
        return
    except ModuleNotFoundError:
        pass
    # repo checkout? put src/ on the path
    for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (cand / "src" / "pinn_sfr_transient").exists():
            sys.path.insert(0, str(cand / "src"))
            importlib.invalidate_caches()
            return
    # otherwise (e.g. Google Colab): install from GitHub. Colab already ships
    # numpy/scipy/matplotlib, so on Colab use --no-deps -- upgrading them inside
    # a live kernel corrupts numpy's ABI ("cannot import name '_center'"). If you
    # hit that error, numpy was upgraded earlier this session: restart the runtime
    # (Runtime > Restart session) and run again.
    cmd = [sys.executable, "-m", "pip", "install"]
    if "google.colab" in sys.modules:
        cmd.append("--no-deps")
    cmd.append(_REPO)
    print(f"Installing pinn-sfr-transient from {_REPO} ...")
    subprocess.run(cmd, check=True)
    importlib.invalidate_caches()


_ensure_package()

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import pinn_sfr_transient
from pinn_sfr_transient import SFRParams, solve_reference
from pinn_sfr_transient.physics import reactivity, void_fraction, flow_fraction

print("pinn_sfr_transient", pinn_sfr_transient.__version__)

## 1. A peak-metrics helper

The two outputs that matter for safety are the **peak power** (the prompt
excursion) and the **peak fuel temperature** (the integrity limit). The helper
below runs one stiff transient for an arbitrary set of `SFRParams` overrides and
returns both.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from pinn_sfr_transient import SFRParams, solve_reference
from pinn_sfr_transient.physics import reactivity


def metrics(n_out=400, **overrides):
    """Peak power and peak fuel temperature for a parameter set."""
    p = SFRParams(**overrides)
    tr = solve_reference(p, n_out=n_out)
    return tr.P.max(), tr.Tf.max()


p0 = SFRParams()
P0, Tf0 = metrics()
print(f"nominal:  alpha_void={p0.alpha_void:.0e}  tau_pump={p0.tau_pump:.0f}s"
      f"  ->  peak power {P0:.2f}x,  peak T_f {Tf0:.0f} K")

## 2. One-parameter sweeps

Two knobs dominate the void-vs-Doppler race: the **sodium void coefficient**
`alpha_void` (how much positive reactivity boiling injects) and the **pump
coast-down time** `tau_pump` (how fast flow is lost). Sweep each on its own
first.

In [ ]:
av_grid = np.linspace(2e-3, 1.1e-2, 25)
tp_grid = np.linspace(2.0, 14.0, 25)

peak_av = [metrics(alpha_void=float(a))[0] for a in av_grid]
peak_tp = [metrics(tau_pump=float(t))[0] for t in tp_grid]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(av_grid * 1e3, peak_av, color="#b22222", lw=2)
ax[0].axvline(p0.alpha_void * 1e3, color="grey", ls=":", label="nominal")
ax[0].set(title="Peak power vs void coefficient",
          xlabel=r"$\alpha_\mathrm{void}$  [10$^{-3}$]", ylabel="peak P / P_nom")
ax[0].legend()
ax[1].plot(tp_grid, peak_tp, color="#16a085", lw=2)
ax[1].axvline(p0.tau_pump, color="grey", ls=":", label="nominal")
ax[1].set(title="Peak power vs pump coast-down",
          xlabel=r"$\tau_\mathrm{pump}$  [s]", ylabel="peak P / P_nom")
ax[1].legend()
fig.tight_layout()
plt.show()

## 3. The 2-D safety map

The interesting behaviour lives in the *joint* dependence. Sweeping both
parameters gives a peak-power map: a near-vertical gradient in `alpha_void` (the
dominant driver) with a weaker `tau_pump` tilt. The white contours mark
excursion multiples; the nominal design point sits in the benign corner.

In [ ]:
av = np.linspace(2e-3, 1.1e-2, 16)
tp = np.linspace(2.0, 14.0, 16)
Ppeak = np.empty((tp.size, av.size))
for i, tpi in enumerate(tp):
    for j, avj in enumerate(av):
        Ppeak[i, j], _ = metrics(alpha_void=float(avj), tau_pump=float(tpi))

fig, ax = plt.subplots(figsize=(9, 6))
pc = ax.pcolormesh(av * 1e3, tp, Ppeak, shading="gouraud", cmap="inferno")
cb = fig.colorbar(pc, ax=ax)
cb.set_label("peak power  P_max / P_nom")
cs = ax.contour(av * 1e3, tp, Ppeak, levels=[1.2, 1.5, 2.0, 2.5],
                colors="white", linewidths=1)
ax.clabel(cs, fmt="%.1f x", fontsize=9)
ax.plot(p0.alpha_void * 1e3, p0.tau_pump, "o", color="cyan", ms=10,
        mec="k", label="nominal")
ax.set(title="ULOF peak-power safety map",
       xlabel=r"sodium void coeff  $\alpha_\mathrm{void}$  [10$^{-3}$]",
       ylabel=r"pump coast-down  $\tau_\mathrm{pump}$  [s]")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

## 4. Phase portrait — power vs net reactivity

Plotting power against net reactivity (in dollars), parameterised by time, turns
each transient into a loop: flow loss drives reactivity positive (void), power
swings up, fuel heating then pulls reactivity negative (Doppler), and the system
spirals back. A larger `alpha_void` simply inflates the loop.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
for av_i, c in zip([4e-3, 6e-3, 8e-3, 1.0e-2],
                   ["#2980b9", "#16a085", "#d35400", "#b22222"]):
    p = SFRParams(alpha_void=av_i)
    tr = solve_reference(p, n_out=1500)
    rho = reactivity(tr.Tf, tr.Tc, p) / p.beta_eff
    ax.plot(rho, tr.P, color=c, lw=1.8,
            label=f"alpha_void={av_i:.0e}  (peak {tr.P.max():.2f}x)")
    ax.plot(rho[0], tr.P[0], "o", color=c, ms=5)
ax.axvline(0, color="grey", ls=":")
ax.axhline(1, color="grey", ls=":")
ax.set(title="ULOF phase portrait: power vs net reactivity",
       xlabel=r"net reactivity  $\rho$  [\$]", ylabel="power  P / P_nom")
ax.legend()
fig.tight_layout()
plt.show()

## Takeaways

* **`alpha_void` is the dominant safety parameter** — peak power climbs steeply
  along it while `tau_pump` only tilts the map. This is exactly why the positive
  sodium void coefficient defines the ULOF transient for this reactor class.
* The nominal design sits in the bounded (~1.4×) corner; Doppler still wins.
* Mapping a *family* of transients like this is the motivation for the
  operator-learning extension of the PINN (see
  [`docs/neural_network.md`](../docs/neural_network.md)). Train the PINN in
  [`01_ulof_walkthrough.ipynb`](01_ulof_walkthrough.ipynb) §7.